# Import necessary packages

In [ ]:
import geopandas as gpd
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import matplotlib as mpl

In [ ]:
# Implement this code in order to make any figures always use Times New Roman. Otherwise when you do it in the figures the matplotlib default can override it
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12

# Set figure titles to be the same
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['axes.titlepad'] = 20
mpl.rcParams['font.family'] = 'Times New Roman'

## Set up base paths

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
networks_folder = base_path / "Processed_data/networks"

# Define the networks_catchments_intersections directory path
networks_catchments_intersections = base_path / "Processed_data/networks/networks_catchments_intersections"
networks_EAELs = base_path / "Processed_data/direct_damages_summary_uids"

In [ ]:
hydrobasins_path = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
catchments_gdf = gpd.read_file(hydrobasins_path)
print("Hydrobasins CRS:", catchments_gdf.crs)

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

# Set up output path

In [ ]:
networks_catchments_EAELs = base_path / "Processed_data/baseline_network_catchment_EAELs"

# Create the directory (and any necessary parent directories), if it doesn't already exist
# networks_catchments_EAELs.mkdir(parents=True, exist_ok=True)

# print(f"Directory created at: {networks_catchments_EAELs}")

map_networks_damages_catchments_figures_folder = base_path / "Results/Map_Figures/networks_damages_catchments"

# Create the directory (and any necessary parent directories), if it doesn't already exist
# map_networks_damages_catchments_figures_folder.mkdir(parents=True, exist_ok=True)

# print(f"Directory created at: {map_networks_damages_catchments_figures_folder}")


## Read in files 

In [ ]:
roads_edges_catchments_intersection = networks_catchments_intersections / "roads_edges_catchments_intersection.gpkg"
roads_nodes_catchments_intersection = networks_catchments_intersections / "roads_nodes_catchments_intersection.gpkg"

# Read the file into a GeoDataFrame
roads_edges_catchments_intersection = gpd.read_file(roads_edges_catchments_intersection)
roads_nodes_catchments_intersection = gpd.read_file(roads_nodes_catchments_intersection)

In [ ]:
print("roads_edges_catchments_intersection CRS:", roads_edges_catchments_intersection.crs)

In [ ]:
roads_edges_catchments_intersection.columns

In [ ]:
roads_edges_catchments_intersection

In [ ]:
roads_edges_EAELs = networks_EAELs / "roads_edges_EAD_EAEL.parquet"
roads_nodes_EAELs = networks_EAELs / "roads_nodes_EAD_EAEL.parquet"

# Instead of using gpd.read_parquet, load the file with pandas:
roads_edges_EAELs_df = pd.read_parquet(roads_edges_EAELs)
roads_nodes_EAELs_df = pd.read_parquet(roads_nodes_EAELs)

In [ ]:
display(roads_edges_EAELs_df)

## Jamaica CRS and boundary 

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

## Check how the networks_catchments intersections look

In [ ]:
roads_edges_catchments_intersection.head()

In [ ]:
roads_nodes_catchments_intersection.head()

## Filter for fluvial baseline 

In [ ]:
# Apply filters
roads_edges_EAELs_fluvial_filtered = roads_edges_EAELs_df.query('hazard == "fluvial" and rcp == "baseline"')
roads_nodes_EAELs_fluvial_filtered = roads_nodes_EAELs_df.query('hazard == "fluvial" and rcp == "baseline"')

In [ ]:
roads_edges_EAELs_fluvial_filtered

In [ ]:
roads_nodes_EAELs_fluvial_filtered

# Edges

### Create a copy of the filtered dataframe and change the asset_id values so that they match between networks_catchments and EAEL layers 

In [ ]:
# Create an explicit copy of the filtered DataFrame
roads_edges_EAELs_fluvial_filtered = roads_edges_EAELs_fluvial_filtered.copy()

# Now safely adjust the edge_id values using .loc
roads_edges_EAELs_fluvial_filtered.loc[:, "edge_id"] = roads_edges_EAELs_fluvial_filtered["edge_id"].str.replace("roadse", "roade", regex=False)

print("Unique edge_id values in EAELs fluvial filtered (after adjustment):")
print(roads_edges_EAELs_fluvial_filtered["edge_id"].unique())

## Merge networks_catchments and the edges_EAELs data 

In [ ]:
# Merge the two layers based on the adjusted 'edge_id'
merged_edges = roads_edges_catchments_intersection.merge(
    roads_edges_EAELs_fluvial_filtered,
    on="edge_id",
    how="left"
)

In [ ]:
print("Merged columns:", merged_edges.columns)
print("Merged GeoDataFrame type:", type(merged_edges))

In [ ]:
# Show all columns in the output
pd.set_option('display.max_columns', None)
display(merged_edges)

### Summarise network information and damages by Hybas ID 

In [ ]:
# # Build aggregation dictionary and group by HYBAS_ID
# cols_to_agg = [
#     'EAD_undefended_amin', 
#     'EAD_undefended_mean', 
#     'EAD_undefended_amax',
#     'EAEL_undefended_amin', 
#     'EAEL_undefended_mean', 
#     'EAEL_undefended_amax'
# ]

# agg_dict = {
#     'edge_id': 'count',
#     'length_m': 'sum'
# }
# for col in cols_to_agg:
#     agg_dict[col] = ['sum']

# summary = merged_edges.groupby("HYBAS_ID").agg(agg_dict).reset_index()

# # Flatten MultiIndex columns
# summary.columns = [
#     '_'.join(filter(None, col)) if isinstance(col, tuple) else col
#     for col in summary.columns
# ]

# # Rename columns for clarity
# summary = summary.rename(columns={
#     'edge_id_count': 'count_edges',
#     'length_m_sum': 'total_length_m'
# })

# # Sort by the numeric value before formatting
# summary_EAD = summary[['HYBAS_ID', 'EAD_undefended_amax_sum']].merge(
#     catchments_gdf[['HYBAS_ID', 'geometry']], on='HYBAS_ID', how='left'
# )

# # Sort using the numeric column directly
# top5_EAD = summary_EAD.sort_values(by='EAD_undefended_amax_sum', ascending=False).head(5)
# display(top5_EAD)

# # Optionally, now format the numbers for display after sorting:
# def format_number(x, decimals=2):
#     try:
#         if isinstance(x, (int, float)) and float(x).is_integer():
#             return f"{int(x):,}"
#         else:
#             return f"{x:,.{decimals}f}"
#     except Exception as e:
#         return x

# for col in summary.columns:
#     if col != 'HYBAS_ID':
#         summary[col] = summary[col].apply(format_number)

# # Continue with saving or further processing
# summary.to_csv("summary_table.csv", index=False)

In [ ]:
# Build aggregation dictionary and group by HYBAS_ID
cols_to_agg = [
    'EAD_undefended_amin', 
    'EAD_undefended_mean', 
    'EAD_undefended_amax',
    'EAEL_undefended_amin', 
    'EAEL_undefended_mean', 
    'EAEL_undefended_amax'
]

agg_dict = {
    'edge_id': 'count',
    'length_m': 'sum'
}
for col in cols_to_agg:
    agg_dict[col] = ['sum']

summary = merged_edges.groupby("HYBAS_ID").agg(agg_dict).reset_index()

# Flatten MultiIndex columns
summary.columns = [
    '_'.join(filter(None, col)) if isinstance(col, tuple) else col
    for col in summary.columns
]

# Rename columns for clarity
summary = summary.rename(columns={
    'edge_id_count': 'count_edges',
    'length_m_sum': 'total_length_m'
})

# Merge the entire summary with catchments to add the geometry.
# This will include all aggregated columns.
summary_with_geom = summary.merge(
    catchments_gdf[['HYBAS_ID', 'geometry']], on='HYBAS_ID', how='left'
)

# Display and export the full summary table
display(summary)
summary.to_csv("roads_edges_baseline_EAD_EAEL_summary_table.csv", index=False)

In [ ]:
# Top 5 HYBAS_IDs based on highest EAD_undefended_amax_sum
top5_EAD = summary_with_geom.sort_values(by='EAD_undefended_amax_sum', ascending=False).head(5)
display(top5_EAD[['HYBAS_ID', 'EAD_undefended_amax_sum', 'geometry']])

In [ ]:
# Top 5 HYBAS_IDs based on highest EAEL_undefended_amax_sum
top5_EAEL = summary_with_geom.sort_values(by='EAEL_undefended_amax_sum', ascending=False).head(5)
display(top5_EAEL[['HYBAS_ID', 'EAEL_undefended_amax_sum', 'geometry']])

In [ ]:
# Define common extents (same as in your working maps)
common_xlim = (593635.9271443005, 848114.2122774671)
common_ylim = (612896.835085367, 712816.0327676072)

fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot background catchments and boundary for context
catchments_gdf.plot(ax=ax, color="lightgrey", edgecolor="white")
jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# Set up colormap (using Reds) based on the EAD damage values
cmap = plt.get_cmap("Reds")
norm = mcolors.Normalize(
    vmin=top5_EAD['EAD_undefended_amax_sum'].min(), 
    vmax=top5_EAD['EAD_undefended_amax_sum'].max()
)

legend_handles = []
for idx, row in top5_EAD.iterrows():
    # Since the value is already numeric, just use it directly
    value = row['EAD_undefended_amax_sum']
    color = cmap(norm(value))
    # Plot the catchment polygon
    gpd.GeoSeries(row['geometry']).plot(ax=ax, color=color, edgecolor="black", linewidth=1.5, zorder=101)
    centroid = row['geometry'].centroid
    # Format the value as desired in the annotation (for example, with 2 decimals)
    ax.annotate(f"{row['HYBAS_ID']}\n{value:,.2f}", 
                xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
    patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({value:,.2f})")
    legend_handles.append(patch)

ax.legend(handles=legend_handles, title="HYBAS ID and baseline EAD undefended max (J$)", 
          bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
          frameon=False, fontsize=12, title_fontsize=14)

# Define scale bar and north arrow functions
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
    x, y = location
    bar_half_length = 0.05
    ax.plot([x - bar_half_length, x + bar_half_length], [y, y],
            transform=ax.transAxes, color="black", linewidth=linewidth)
    for pos in [x - bar_half_length, x, x + bar_half_length]:
        ax.plot([pos, pos], [y - tick_height/2, y + tick_height/2],
                transform=ax.transAxes, color="black", linewidth=linewidth)
    ax.text(x - bar_half_length, y - tick_height - label_offset, "0",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x, y - tick_height - label_offset, f"{int(length_km // 2)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length, y - tick_height - label_offset, f"{int(length_km)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length + km_offset, y, "km",
            transform=ax.transAxes, ha="left", va="center", fontsize=12)

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
    x, y = location
    ax.annotate("", xy=(x, y + size), xycoords="axes fraction",
                xytext=(x, y), textcoords="axes fraction",
                arrowprops=dict(facecolor="black", edgecolor="black", headwidth=10, headlength=15, width=5))
    ax.text(x, y + size + label_offset, "N",
            transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

# Add scale bar and north arrow
add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
add_north_arrow(ax, location=(0.9, 0.85))

ax.set_xlim(common_xlim)
ax.set_ylim(common_ylim)
ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
ax.set_title("Top 5 Catchments by highest baseline EAD undefended max Value", 
             fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

plt.tight_layout()
fig.savefig(map_networks_damages_catchments_figures_folder / "roads_top5_baseline_EAD_max.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot background catchments and boundary for context
catchments_gdf.plot(ax=ax, color="lightgrey", edgecolor="white")
jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# Set up colormap (using Reds) based on the EAEL damage values
cmap = plt.get_cmap("Reds")
norm = mcolors.Normalize(
    vmin=top5_EAEL['EAEL_undefended_amax_sum'].min(), 
    vmax=top5_EAEL['EAEL_undefended_amax_sum'].max()
)

legend_handles = []
for idx, row in top5_EAEL.iterrows():
    # Use the numeric value directly
    value = row['EAEL_undefended_amax_sum']
    color = cmap(norm(value))
    # Plot the catchment polygon
    gpd.GeoSeries(row['geometry']).plot(ax=ax, color=color, edgecolor="black", linewidth=1.5, zorder=101)
    centroid = row['geometry'].centroid
    ax.annotate(f"{row['HYBAS_ID']}\n{value:,.2f}", 
                xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
    patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({value:,.2f})")
    legend_handles.append(patch)

ax.legend(handles=legend_handles, title="HYBAS ID and EAEL undefended max (J$)", 
          bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
          frameon=False, fontsize=12, title_fontsize=14)

# Define scale bar and north arrow functions
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
    x, y = location
    bar_half_length = 0.05
    ax.plot([x - bar_half_length, x + bar_half_length], [y, y],
            transform=ax.transAxes, color="black", linewidth=linewidth)
    for pos in [x - bar_half_length, x, x + bar_half_length]:
        ax.plot([pos, pos], [y - tick_height/2, y + tick_height/2],
                transform=ax.transAxes, color="black", linewidth=linewidth)
    ax.text(x - bar_half_length, y - tick_height - label_offset, "0",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x, y - tick_height - label_offset, f"{int(length_km // 2)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length, y - tick_height - label_offset, f"{int(length_km)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length + km_offset, y, "km",
            transform=ax.transAxes, ha="left", va="center", fontsize=12)

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
    x, y = location
    ax.annotate("", xy=(x, y + size), xycoords="axes fraction",
                xytext=(x, y), textcoords="axes fraction",
                arrowprops=dict(facecolor="black", edgecolor="black", headwidth=10, headlength=15, width=5))
    ax.text(x, y + size + label_offset, "N",
            transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

# Add scale bar and north arrow
add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
add_north_arrow(ax, location=(0.9, 0.85))

ax.set_xlim(common_xlim)
ax.set_ylim(common_ylim)
ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
ax.set_title("Top 5 Catchments by highest baseline EAEL undefended max Value", 
             fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

plt.tight_layout()
fig.savefig(map_networks_damages_catchments_figures_folder / "roads_top5_baseline_EAEL_max.png", dpi=300, bbox_inches="tight")
plt.show()

### If we want to plot it on a map we have to merge again with the original catchments layer. 
#### My previous geometries are of the specific edges not the catchments 

In [ ]:

# # Define common extents (same as in your working maps)
# common_xlim = (593635.9271443005, 848114.2122774671)
# common_ylim = (612896.835085367, 712816.0327676072)

# fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# # Plot the background catchments and boundary for context
# catchments_gdf.plot(ax=ax, color="lightgrey", edgecolor="white")
# jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# # Set up a colormap (using Reds) based on the EAD damage values
# cmap = plt.get_cmap("Reds")
# norm = mcolors.Normalize(
#     vmin=top5_EAD['EAD_undefended_amax_sum'].str.replace(',', '').astype(float).min(), 
#     vmax=top5_EAD['EAD_undefended_amax_sum'].str.replace(',', '').astype(float).max()
# )

# legend_handles = []
# for idx, row in top5_EAD.iterrows():
#     # Convert the string value to float
#     value = float(row['EAD_undefended_amax_sum'].replace(',', ''))
#     color = cmap(norm(value))
#     # Plot the catchment polygon
#     gpd.GeoSeries(row['geometry']).plot(ax=ax, color=color, edgecolor="black", linewidth=1.5, zorder=101)
#     centroid = row['geometry'].centroid
#     ax.annotate(f"{row['HYBAS_ID']}\n{row['EAD_undefended_amax_sum']}", 
#                 xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
#     patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({row['EAD_undefended_amax_sum']})")
#     legend_handles.append(patch)

# ax.legend(handles=legend_handles, title="HYBAS ID and EAD_undefended_amax", 
#           bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
#           frameon=False, fontsize=12, title_fontsize=14)

# # Define scale bar and north arrow functions
# def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
#     x, y = location
#     bar_half_length = 0.05
#     ax.plot([x - bar_half_length, x + bar_half_length], [y, y],
#             transform=ax.transAxes, color="black", linewidth=linewidth)
#     for pos in [x - bar_half_length, x, x + bar_half_length]:
#         ax.plot([pos, pos], [y - tick_height/2, y + tick_height/2],
#                 transform=ax.transAxes, color="black", linewidth=linewidth)
#     ax.text(x - bar_half_length, y - tick_height - label_offset, "0",
#             transform=ax.transAxes, ha="center", va="center", fontsize=10)
#     ax.text(x, y - tick_height - label_offset, f"{int(length_km // 2)}",
#             transform=ax.transAxes, ha="center", va="center", fontsize=10)
#     ax.text(x + bar_half_length, y - tick_height - label_offset, f"{int(length_km)}",
#             transform=ax.transAxes, ha="center", va="center", fontsize=10)
#     ax.text(x + bar_half_length + km_offset, y, "km",
#             transform=ax.transAxes, ha="left", va="center", fontsize=12)

# def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
#     x, y = location
#     ax.annotate("", xy=(x, y + size), xycoords="axes fraction",
#                 xytext=(x, y), textcoords="axes fraction",
#                 arrowprops=dict(facecolor="black", edgecolor="black", headwidth=10, headlength=15, width=5))
#     ax.text(x, y + size + label_offset, "N",
#             transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

# add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
# add_north_arrow(ax, location=(0.9, 0.85))

# ax.set_xlim(common_xlim)
# ax.set_ylim(common_ylim)
# ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
# ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
# ax.set_title("Top 5 Catchments by EAD_undefended_amax Value", 
#              fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

# plt.tight_layout()
# fig.savefig(map_networks_damages_catchments_figures_folder / "roadstest.png", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# # Merge the aggregated summary with catchments to get geometry for plotting.
# summary_EAEL = summary[['HYBAS_ID', 'EAEL_undefended_amax_sum']].merge(
#     catchments_gdf[['HYBAS_ID', 'geometry']], on='HYBAS_ID', how='left'
# )

# # Sort by EAEL_undefended_amax_sum descending and select the top 5
# top5_EAEL = summary_EAEL.sort_values(by='EAEL_undefended_amax_sum', ascending=False).head(5)

# fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# # Plot the background catchments and boundary for context
# catchments_gdf.plot(ax=ax, color="lightgrey", edgecolor="white")
# jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# # Set up a colormap (using Blues) based on the EAEL damage values
# cmap = plt.get_cmap("Blues")
# norm = mcolors.Normalize(
#     vmin=top5_EAEL['EAEL_undefended_amax_sum'].str.replace(',', '').astype(float).min(), 
#     vmax=top5_EAEL['EAEL_undefended_amax_sum'].str.replace(',', '').astype(float).max()
# )

# legend_handles = []
# for idx, row in top5_EAEL.iterrows():
#     value = float(row['EAEL_undefended_amax_sum'].replace(',', ''))
#     color = cmap(norm(value))
#     # Plot the catchment polygon
#     gpd.GeoSeries(row['geometry']).plot(ax=ax, color=color, edgecolor="black", linewidth=1.5, zorder=101)
#     centroid = row['geometry'].centroid
#     ax.annotate(f"{row['HYBAS_ID']}\n{row['EAEL_undefended_amax_sum']}", 
#                 xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
#     patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({row['EAEL_undefended_amax_sum']})")
#     legend_handles.append(patch)

# ax.legend(handles=legend_handles, title="HYBAS ID and EAEL_undefended_amax", 
#           bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
#           frameon=False, fontsize=12, title_fontsize=14)

# # Use the same scale bar and north arrow functions
# add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
# add_north_arrow(ax, location=(0.9, 0.85))

# ax.set_xlim(common_xlim)
# ax.set_ylim(common_ylim)
# ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
# ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
# ax.set_title("Top 5 Catchments by EAEL_undefended_amax Value", 
#              fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

# plt.tight_layout()
# fig.savefig(map_networks_damages_catchments_figures_folder / "roads(edges)_EAEL_max_value_by_catchment_map.png", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# # -----------------------------
# # Function to Compute Catchment Damage Statistics
# # -----------------------------
# def compute_catchment_damage_stats(merged_edges, catchments_gdf, damage_col):
#     """
#     Given a damage column (e.g., 'EAD_undefended_amax' or 'EAEL_undefended_amax'),
#     create a flag for damaged edges, calculate the damaged road length, aggregate by HYBAS_ID,
#     and compute the proportion of the catchment's road length that is damaged.
#     """
#     flag = damage_col + '_damaged'
#     merged_edges[flag] = merged_edges[damage_col].fillna(0) > 0
#     damaged_length_col = 'damaged_length_m_' + damage_col
#     merged_edges[damaged_length_col] = merged_edges['length_m'].where(merged_edges[flag], 0)
    
#     # Aggregate total and damaged lengths by HYBAS_ID
#     length_stats_numeric = merged_edges.groupby("HYBAS_ID").agg(
#         total_length_m=('length_m', 'sum'),
#         damaged_length_m=(damaged_length_col, 'sum')
#     ).reset_index()
#     length_stats_numeric['damaged_length_m'] = length_stats_numeric['damaged_length_m'].round(0)
    
#     # Merge these stats with the catchments GeoDataFrame
#     catchments_damage = catchments_gdf.merge(length_stats_numeric, on="HYBAS_ID", how="left")
#     catchments_damage['damaged_length_m'] = catchments_damage['damaged_length_m'].fillna(0)
#     catchments_damage['total_length_m'] = catchments_damage['total_length_m'].fillna(0)
    
#     # Calculate percentage of damaged road length within the catchment
#     catchments_damage["proportion_damaged"] = (
#         catchments_damage["damaged_length_m"] / catchments_damage["total_length_m"] * 100
#     )
#     catchments_damage["proportion_damaged"] = catchments_damage["proportion_damaged"].fillna(0)
#     return catchments_damage

# # -----------------------------
# # Compute Damage Stats for Both Metrics
# # -----------------------------
# # Use .copy() to avoid modifying the original merged_edges
# catchments_damage_EAD = compute_catchment_damage_stats(merged_edges.copy(), catchments_gdf, 'EAD_undefended_amax')
# catchments_damage_EAEL = compute_catchment_damage_stats(merged_edges.copy(), catchments_gdf, 'EAEL_undefended_amax')


#### Map 1: EAD – Top 5 Catchments by Damaged Road Length (m)

In [ ]:
# # Define common extents (same as your working maps)
# common_xlim = (593635.9271443005, 848114.2122774671)
# common_ylim = (612896.835085367, 712816.0327676072)

# # Get the top 5 catchments for EAD by absolute damaged length
# top5_damaged_length_EAD = catchments_damage_EAD.sort_values(by="damaged_length_m", ascending=False).head(5)

# fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# # Plot the base layer
# catchments_damage_EAD.plot(ax=ax, color="lightgrey", edgecolor="white")
# jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# # Set up colormap
# cmap = plt.get_cmap("Reds")
# norm = mcolors.Normalize(vmin=top5_damaged_length_EAD["damaged_length_m"].min(), 
#                          vmax=top5_damaged_length_EAD["damaged_length_m"].max())

# legend_handles = []
# for idx, row in top5_damaged_length_EAD.iterrows():
#     color = cmap(norm(row["damaged_length_m"]))
#     # Plot the catchment
#     gpd.GeoSeries(row["geometry"]).plot(ax=ax, color=color, edgecolor="black", linewidth=1.5, zorder=101)
#     centroid = row["geometry"].centroid
#     ax.annotate(f"{row['HYBAS_ID']}\n{int(row['damaged_length_m']):,} m", 
#                 xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
#     patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({int(row['damaged_length_m']):,} m)")
#     legend_handles.append(patch)

# # Add legend
# ax.legend(handles=legend_handles, title="HYBAS ID and Damaged Road Length", 
#           bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
#           frameon=False, fontsize=12, title_fontsize=14)

# # Define scale bar and north arrow functions (reuse these)
# def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
#     x, y = location
#     bar_half_length = 0.05
#     ax.plot([x - bar_half_length, x + bar_half_length], [y, y],
#             transform=ax.transAxes, color="black", linewidth=linewidth)
#     for pos in [x - bar_half_length, x, x + bar_half_length]:
#         ax.plot([pos, pos], [y - tick_height/2, y + tick_height/2],
#                 transform=ax.transAxes, color="black", linewidth=linewidth)
#     ax.text(x - bar_half_length, y - tick_height - label_offset, "0",
#             transform=ax.transAxes, ha="center", va="center", fontsize=10)
#     ax.text(x, y - tick_height - label_offset, f"{int(length_km // 2)}",
#             transform=ax.transAxes, ha="center", va="center", fontsize=10)
#     ax.text(x + bar_half_length, y - tick_height - label_offset, f"{int(length_km)}",
#             transform=ax.transAxes, ha="center", va="center", fontsize=10)
#     ax.text(x + bar_half_length + km_offset, y, "km",
#             transform=ax.transAxes, ha="left", va="center", fontsize=12)

# def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
#     x, y = location
#     ax.annotate("", xy=(x, y + size), xycoords="axes fraction",
#                 xytext=(x, y), textcoords="axes fraction",
#                 arrowprops=dict(facecolor="black", edgecolor="black", headwidth=10, headlength=15, width=5))
#     ax.text(x, y + size + label_offset, "N",
#             transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

# add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
# add_north_arrow(ax, location=(0.9, 0.85))

# # Set common extents, labels, and title
# ax.set_xlim(common_xlim)
# ax.set_ylim(common_ylim)
# ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
# ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
# ax.set_title("Top 5 Catchments according to Damaged Road Length (m) - baseline EAD undefended max", 
#              fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

# plt.tight_layout()
# fig.savefig(map_networks_damages_catchments_figures_folder / "roads(edges)_EAD_max_length_by_catchment_map.png", dpi=300, bbox_inches="tight")
# plt.show()

#### Map 2: EAD – Top 5 Catchments by Damaged Road Proportion (%)

In [ ]:
# # Get the top 5 catchments for EAD by proportion damaged
# top5_proportion_EAD = catchments_damage_EAD.sort_values(by="proportion_damaged", ascending=False).head(5)

# fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# # Plot base layers
# catchments_damage_EAD.plot(ax=ax, color="lightgrey", edgecolor="white")
# jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# cmap = plt.get_cmap("Reds")
# norm = mcolors.Normalize(vmin=top5_proportion_EAD["proportion_damaged"].min(), 
#                          vmax=top5_proportion_EAD["proportion_damaged"].max())

# legend_handles = []
# for idx, row in top5_proportion_EAD.iterrows():
#     color = cmap(norm(row["proportion_damaged"]))
#     gpd.GeoSeries(row["geometry"]).plot(ax=ax, color=color, edgecolor="black", linewidth=1.5, zorder=101)
#     centroid = row["geometry"].centroid
#     ax.annotate(f"{row['HYBAS_ID']}\n{row['proportion_damaged']:.0f}%", 
#                 xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
#     patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({row['proportion_damaged']:.0f}%)")
#     legend_handles.append(patch)

# ax.legend(handles=legend_handles, title="HYBAS ID and percentage of road length damaged", 
#           bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
#           frameon=False, fontsize=12, title_fontsize=14)

# add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
# add_north_arrow(ax, location=(0.9, 0.85))

# ax.set_xlim(common_xlim)
# ax.set_ylim(common_ylim)
# ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
# ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
# ax.set_title("Top 5 Catchments by the percentage of roads damaged - baseline EAD undefended max", 
#              fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

# plt.tight_layout()
# fig.savefig(map_networks_damages_catchments_figures_folder / "roads(edges)_EAD_max_percentage_by_catchment_map.png", dpi=300, bbox_inches="tight")
# plt.show()

#### Map 3: EAEL – Top 5 Catchments by Damaged Road Length (m)

In [ ]:
# # Get the top 5 catchments for EAEL by absolute damaged length
# top5_damaged_length_EAEL = catchments_damage_EAEL.sort_values(by="damaged_length_m", ascending=False).head(5)

# fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# # Plot base layers
# catchments_damage_EAEL.plot(ax=ax, color="lightgrey", edgecolor="white")
# jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# cmap = plt.get_cmap("Blues")
# norm = mcolors.Normalize(vmin=top5_damaged_length_EAEL["damaged_length_m"].min(), 
#                          vmax=top5_damaged_length_EAEL["damaged_length_m"].max())

# legend_handles = []
# for idx, row in top5_damaged_length_EAEL.iterrows():
#     color = cmap(norm(row["damaged_length_m"]))
#     gpd.GeoSeries(row["geometry"]).plot(ax=ax, color=color, edgecolor="black", linewidth=1.5, zorder=101)
#     centroid = row["geometry"].centroid
#     ax.annotate(f"{row['HYBAS_ID']}\n{int(row['damaged_length_m']):,} m", 
#                 xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
#     patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({int(row['damaged_length_m']):,} m)")
#     legend_handles.append(patch)

# ax.legend(handles=legend_handles, title="HYBAS ID and Damaged Length", 
#           bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
#           frameon=False, fontsize=12, title_fontsize=14)

# add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
# add_north_arrow(ax, location=(0.9, 0.85))

# ax.set_xlim(common_xlim)
# ax.set_ylim(common_ylim)
# ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
# ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
# ax.set_title("Top 5 Catchments according to Damaged Road Length (m) - baseline EAEL undefended max", 
#              fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

# plt.tight_layout()
# fig.savefig(map_networks_damages_catchments_figures_folder / "roads(edges)_EAEL_max_length_by_catchment_map.png", dpi=300, bbox_inches="tight")
# plt.show()

#### Map 4: EAEL – Top 5 Catchments by Damaged Road Proportion (%)

In [ ]:
# # Get the top 5 catchments for EAEL by proportion damaged
# top5_proportion_EAEL = catchments_damage_EAEL.sort_values(by="proportion_damaged", ascending=False).head(5)

# fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# # Plot base layers
# catchments_damage_EAEL.plot(ax=ax, color="lightgrey", edgecolor="white")
# jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# cmap = plt.get_cmap("Blues")
# norm = mcolors.Normalize(vmin=top5_proportion_EAEL["proportion_damaged"].min(), 
#                          vmax=top5_proportion_EAEL["proportion_damaged"].max())

# legend_handles = []
# for idx, row in top5_proportion_EAEL.iterrows():
#     color = cmap(norm(row["proportion_damaged"]))
#     gpd.GeoSeries(row["geometry"]).plot(ax=ax, color=color, edgecolor="black", linewidth=1.5, zorder=101)
#     centroid = row["geometry"].centroid
#     ax.annotate(f"{row['HYBAS_ID']}\n{row['proportion_damaged']:.0f}%", 
#                 xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
#     patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({row['proportion_damaged']:.0f}%)")
#     legend_handles.append(patch)

# ax.legend(handles=legend_handles, title="HYBAS ID and % of roads with EAELs", 
#           bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
#           frameon=False, fontsize=12, title_fontsize=14)

# add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
# add_north_arrow(ax, location=(0.9, 0.85))

# ax.set_xlim(common_xlim)
# ax.set_ylim(common_ylim)
# ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
# ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
# ax.set_title("Top 5 Catchments by percentage of roads with EAELs - baseline EAEL undefended max", 
#              fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

# plt.tight_layout()
# fig.savefig(map_networks_damages_catchments_figures_folder / "roads(edges)_baseline_EAEL_max_percentage_by_catchment_map.png", dpi=300, bbox_inches="tight")
# plt.show()

# Nodes

## Merge networks_catchments and the nodes_EAELs data 

In [ ]:
# Now, merge the catchments nodes with the EAELs nodes on the common "node_id" field.
merged_nodes = roads_nodes_catchments_intersection.merge(
    roads_nodes_EAELs_fluvial_filtered,
    on="node_id",
    how="left",
    suffixes=('_catchments', '_EAELs')
)

In [ ]:
# If you notice that EAELs node IDs use the prefix "roadsn" but catchments use "roadn",
# adjust the EAELs node_id values. (If they already match, you can skip this step.)
roads_nodes_EAELs_fluvial_filtered = roads_nodes_EAELs_fluvial_filtered.copy()
roads_nodes_EAELs_fluvial_filtered.loc[:, "node_id"] = roads_nodes_EAELs_fluvial_filtered["node_id"].str.replace("roadsn", "roadn", regex=False)

# Verify the replacement
print("Adjusted unique node_id values in EAELs fluvial filtered:")
print(roads_nodes_EAELs_fluvial_filtered["node_id"].unique())

In [ ]:
# Display the first few rows of the merged nodes GeoDataFrame
display("Merged nodes:")
display(merged_nodes.head())

In [ ]:
# Define the columns for which we want summary statistics
nodes_cols_to_agg = [
    'EAD_undefended_amin', 
    'EAD_undefended_mean', 
    'EAD_undefended_amax',
    'EAEL_undefended_amin', 
    'EAEL_undefended_mean', 
    'EAEL_undefended_amax'
]

# Build an aggregation dictionary that includes:
# - count_edges: count of edge_id
# - total_length_m: sum of length_m
# - For each EA(E) column: sum, mean, min, and max
agg_dict = {
    'node_id': 'count',
}
for col in nodes_cols_to_agg:
    agg_dict[col] = ['sum']

# Group by HYBAS_ID
nodes_summary = merged_nodes.groupby("HYBAS_ID").agg(agg_dict).reset_index()

# Flatten the MultiIndex columns
nodes_summary.columns = [
    '_'.join(filter(None, col)) if isinstance(col, tuple) else col
    for col in nodes_summary.columns
]

# Rename columns for clarity:
nodes_summary = nodes_summary.rename(columns={
    'node_id_count': 'count_nodes',
})

# Define a helper function to format numbers:
def format_number(x, decimals=2):
    try:
        # If x is numeric and essentially an integer, format without decimals.
        if isinstance(x, (int, float)) and float(x).is_integer():
            return f"{int(x):,}"
        else:
            return f"{x:,.{decimals}f}"
    except:
        return x

# Format all numeric columns (except HYBAS_ID) with commas for thousands.
for col in nodes_summary.columns:
    if col != 'HYBAS_ID':
        nodes_summary[col] = nodes_summary[col].apply(format_number)


# Rename the columns as requested
nodes_summary = nodes_summary.rename(columns={
    "count_nodes": "Number of nodes",
})
# Display the summary
display(nodes_summary)

In [ ]:
# For nodes:

# For nodes: similarly, flag as damaged if EAEL_undefended_mean > 0
merged_nodes['node_damaged'] = merged_nodes['EAEL_undefended_mean'].fillna(0) > 0


nodes_damage_stats = merged_nodes.groupby("HYBAS_ID").agg(
    total_nodes=('node_id', 'count'),
    damaged_nodes=('node_damaged', 'sum')
).reset_index()
nodes_damage_stats['pct_damaged_nodes'] = (
    nodes_damage_stats['damaged_nodes'] / nodes_damage_stats['total_nodes'] * 100
).round(2)
